# NCAA March Madness Feature Engineering Pipeline

This notebook implements the data processing and feature engineering pipeline for predicting NCAA tournament outcomes. It covers:
1. Data Loading
2. Aggregating Regular Season Statistics
3. Calculating Efficiency Ratings
4. Processing Seeds and Rankings
5. Creating the Final Feature Set for Training

In [1]:
import pandas as pd
import numpy as np
import os

DATA_PATH = './march-machine-learning-mania-2026/'

# Load essential files
m_teams = pd.read_csv(os.path.join(DATA_PATH, 'MTeams.csv'))
m_seasons = pd.read_csv(os.path.join(DATA_PATH, 'MSeasons.csv'))
m_seeds = pd.read_csv(os.path.join(DATA_PATH, 'MNCAATourneySeeds.csv'))
m_regular_results = pd.read_csv(os.path.join(DATA_PATH, 'MRegularSeasonDetailedResults.csv'))
m_tourney_results = pd.read_csv(os.path.join(DATA_PATH, 'MNCAATourneyDetailedResults.csv'))
m_rankings = pd.read_csv(os.path.join(DATA_PATH, 'MMasseyOrdinals.csv'))

print("Data loaded successfully.")

Data loaded successfully.


## 1. Feature Engineering: Basic Team Stats

We need to transform game-by-game results into seasonal averages for each team.

In [ ]:
def calculate_team_stats(df):
    # Create columns for winning team stats
    w_cols = ['WScore', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF']
    # Create columns for losing team stats
    l_cols = ['LScore', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']
    
    # Stats for the winner in each game
    win_stats = df[['Season', 'WTeamID'] + w_cols].rename(columns={'WTeamID': 'TeamID'})
    win_stats.columns = [c.replace('W', '') if c != 'Season' else c for c in win_stats.columns]
    
    # Stats for the loser in each game
    lose_stats = df[['Season', 'LTeamID'] + l_cols].rename(columns={'LTeamID': 'TeamID'})
    lose_stats.columns = [c.replace('L', '') if c != 'Season' else c for c in lose_stats.columns]
    
    # Combine all games (each game yields two rows: one for TeamA and one for TeamB)
    combined_stats = pd.concat([win_stats, lose_stats])
    
    # Group by Season and TeamID
    seasonal_stats = combined_stats.groupby(['Season', 'TeamID']).mean().reset_index()
    return seasonal_stats

m_seasonal_stats = calculate_team_stats(m_regular_results)
m_seasonal_stats.head()

## 2. Advanced Metrics: Efficiency Ratings

Efficiency ratings are more predictive than raw points. We use the common formula for possessions:
$Possessions = FGA + 0.475 * FTA - OR + TO$

In [ ]:
def add_efficiency_metrics(df):
    # Possessions estimate
    df['Possessions'] = df['FGA'] + 0.475 * df['FTA'] - df['OR'] + df['TO']
    
    # Offensive Efficiency (Points per 100 possessions)
    df['OffEff'] = (df['Score'] / df['Possessions']) * 100
    
    # Effective Field Goal Percentage
    df['eFG'] = (df['FGM'] + 0.5 * df['FGM3']) / df['FGA']
    
    return df

m_seasonal_stats = add_efficiency_metrics(m_seasonal_stats)
m_seasonal_stats.head()

## 3. Seed and Rankings

We need to clean the Seed strings (e.g., 'W01' -> 1).

In [ ]:
m_seeds['SeedInt'] = m_seeds['Seed'].apply(lambda x: int(x[1:3]))

# Get final regular season rankings (Day 133)
m_final_rankings = m_rankings[m_rankings['RankingDayNum'] == 133].copy()
# Using consensus (or common system like Pomeroy 'POM' if available)
m_consensus_rank = m_final_rankings.groupby(['Season', 'TeamID'])['OrdinalRank'].mean().reset_index().rename(columns={'OrdinalRank': 'AvgRank'})

print("Rankings and Seeds processed.")

## 4. Merging All Features

Merge the seasonal averages, efficiency metrics, seeds, and rankings into a single team-level feature dataframe.

In [ ]:
m_features = m_seasonal_stats.merge(m_seeds[['Season', 'TeamID', 'SeedInt']], on=['Season', 'TeamID'], how='left')
m_features = m_features.merge(m_consensus_rank, on=['Season', 'TeamID'], how='left')

# Fill missing seeds for teams that didn't make the tournament with a high value (e.g., 20)
m_features['SeedInt'] = m_features['SeedInt'].fillna(20)
m_features['AvgRank'] = m_features['AvgRank'].fillna(m_features['AvgRank'].max())

m_features.to_csv('m_team_features.csv', index=False)
print("Final feature set created and saved to m_team_features.csv")